In [1]:
!pip install --upgrade monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.9 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu

In [2]:
import os
import gc
import torch
import random
import numpy as np
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import distance_transform_edt
from monai.transforms import (
    Compose, AsDiscrete,
    RandFlipd, RandRotate90d,
    RandScaleIntensityd, RandShiftIntensityd,
    RandGaussianNoised, RandCropByPosNegLabeld,
    SpatialPadd, Resized
)
from monai.data import decollate_batch, set_track_meta
from monai.inferers import sliding_window_inference
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from monai.losses import TverskyLoss
from monai.metrics import DiceMetric


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
os.environ['PYTHONHASHSEED']       = str(SEED)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
set_track_meta(False)  

import warnings
warnings.filterwarnings("ignore")

2026-06-30 06:55:38.923896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782802539.336262      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782802539.450896      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782802540.407906      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782802540.407946      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782802540.407949      58 computation_placer.cc:177] computation placer alr

In [3]:
def generate_sdf(binary_label, margin=15.0):
    """
    binary_label: numpy array, shape (H, W, D), values {0, 1}
    margin: maximum distance threshold in voxels to truncate
    returns: TSDF numpy array, normalized to [-1, 1], same shape, float32
    """
    dist_outside = distance_transform_edt(1 - binary_label)
    dist_inside  = distance_transform_edt(binary_label)
    
    # Raw SDF map
    sdf = dist_outside - dist_inside
    
    # Clip to a fixed absolute margin instead of an arbitrary batch max
    # This ensures a distance of 3 voxels means the exact same thing everywhere
    sdf = np.clip(sdf, -margin, margin)
    
    # Scale strictly to [-1, 1] using the static margin
    tsdf = sdf / margin
    
    return tsdf.astype(np.float32)

In [5]:
folder = '/kaggle/input/datasets/ayushpatel170/brain-tumor-numpy-processed2/'
categories = os.listdir('/kaggle/input/datasets/ayushpatel170/brain-tumor-numpy-processed2')

train_files = {cat: os.listdir(folder + cat + '/Train') for cat in categories}
val_files = {cat: os.listdir(folder + cat + '/Val') for cat in categories}

In [6]:
class BTData(Dataset):
    def __init__(self, train, image_paths, label_paths):
        # self.image_paths = sorted(image_paths)
        # self.label_paths = sorted(label_paths)
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.train = train
        self.transforms = Compose([
            RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
            RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
            RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
            RandRotate90d(keys=['image', 'label'], prob=0.5),
            RandScaleIntensityd(keys='image', factors=0.2, prob=0.2),
            RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
            RandGaussianNoised(keys="image", std=0.01, prob=0.15),            
        ])
        self.padder = SpatialPadd(keys=['image', 'label'], spatial_size = (128, 128, 128))
        self.cropper = RandCropByPosNegLabeld(
            keys=['image', 'label'],
            label_key = 'label',
            spatial_size = (128, 128, 128),
            pos=2,
            neg=1
        )
        self.resize = Resized(keys=['image', 'label'], spatial_size=(128, 128, 128), mode=['trilinear', 'nearest'])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = self.label_paths[idx]

        image = torch.from_numpy(np.load(image_path).copy()).float()
        label = torch.from_numpy((np.load(label_path).copy() > 0).astype(np.uint8)).long().unsqueeze(0)
        # label = (label > 0).astype(np.uint8)
                
        tumor_type = 0 if 'Glioma' in image_path else \
                     1 if 'Meningioma' in image_path else 2

        data = {'image': image, 'label': label}
        data = self.padder(data)

        if self.train:
            data = self.cropper(data)[0]
            data = self.transforms(data)
            image = data["image"].as_tensor() \
                    if hasattr(data["image"], 'as_tensor') else data["image"]
            label = data["label"].as_tensor() \
                    if hasattr(data["label"], 'as_tensor') else data["label"]
 
            tumor_type = tumor_type if label.sum().item() > 0 else 3            
        
        data_128 = self.resize(data)
        image_128 = data_128["image"]
        label_128 = data_128["label"]

        sdf = generate_sdf(label.squeeze(0).cpu().numpy())
        sdf = torch.from_numpy(sdf).float().unsqueeze(0)  

        return image, label, sdf, tumor_type, image_128, label_128

In [7]:
train_images = []
train_labels = []
val_images = []
val_labels = []

for cat, paths in tqdm(train_files.items()):
    img_list = []
    lbl_list = []
    for filename in paths:
        if 'image.npy' in filename:
            img_list.append(folder + cat + '/Train/' + filename)
        elif 'label.npy' in filename:
            lbl_list.append(folder + cat + '/Train/' + filename)

    img_list = sorted(img_list)
    lbl_list = sorted(lbl_list)
            
    train_images.extend(img_list)
    train_labels.extend(lbl_list)
    
for cat, paths in tqdm(val_files.items()):
    img_list = []
    lbl_list = []
    for filename in paths:
        if 'image.npy' in filename:
            img_list.append(folder + cat + '/Val/' + filename)
        elif 'label.npy' in filename:
            lbl_list.append(folder + cat + '/Val/' + filename)

    img_list = sorted(img_list)
    lbl_list = sorted(lbl_list)
            
    val_images.extend(img_list[:50])
    val_labels.extend(lbl_list[:50])

train_dataset = BTData(True, train_images, train_labels)
val_dataset = BTData(False, val_images, val_labels)


train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4)

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),  
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.skip = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.conv(x) + self.skip(x)


class SkipAttention(nn.Module):
    """
    Multi-head self-attention applied on a skip connection.
    Only used for deep encoder skips where spatial size is small enough:
      enc3 → 8^3  =   512 tokens  (very feasible)
      enc2 → 16^3 = 4,096 tokens  (feasible)
    NOT used for enc1 (32^3 = 32k) or enc0 (64^3 = 262k).
    """
    def __init__(self, channels):
        super().__init__()
        # num_heads must divide channels evenly; scale relative to channel count
        num_heads = max(1, channels // 32)
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True,   # expects (B, seq_len, embed_dim)
            bias=True,          # attention projections need bias (no norm inside MHA)
        )
        self.norm = nn.InstanceNorm3d(channels)

    def forward(self, x):
        B, C, H, W, D = x.shape

        # (B, C, H, W, D) → (B, H*W*D, C)  — treat voxels as sequence tokens
        tokens = x.permute(0, 2, 3, 4, 1).reshape(B, H * W * D, C)

        # self-attention: Q = K = V = tokens
        attn_out, _ = self.attn(tokens, tokens, tokens)

        # (B, H*W*D, C) → (B, C, H, W, D)
        attn_out = attn_out.reshape(B, H, W, D, C).permute(0, 4, 1, 2, 3)

        # residual + norm (skip + refined skip, then normalize)
        return self.norm(x + attn_out)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_attention=False):
        super().__init__()
        self.res  = ResBlock(in_ch, out_ch)
        self.down = nn.Conv3d(out_ch, out_ch, 2, stride=2, bias=False)
        # attention sits on the skip path, applied before the skip is saved
        self.attn = SkipAttention(out_ch) if use_attention else nn.Identity()

    def forward(self, x):
        skip = self.res(x)        # fix: res runs first, producing the skip
        skip = self.attn(skip)    # refine skip with MHSA (no-op for shallow encoders)
        return self.down(skip), skip


class DecoderBlock(nn.Module):
    """
    Addition-based merge: up(x) + skip, then ResBlock.
    With addition, skip channels must equal out_ch —
    which holds here since encoder and decoder are symmetric.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up  = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2, bias=False)
        self.res = ResBlock(out_ch, out_ch)

    def forward(self, x, skip):
        return self.res(self.up(x) + skip)


class BTUnetAttention(nn.Module):
    def __init__(self):
        super().__init__()   # fix: was missing entirely

        # enc0, enc1: shallow — spatial too large for MHSA, no attention
        # enc2, enc3: deep   — use_attention=True (16^3 and 8^3 tokens)
        self.enc0 = EncoderBlock(1,    32,  use_attention=False)
        self.enc1 = EncoderBlock(32,   64,  use_attention=False)
        self.enc2 = EncoderBlock(64,   128, use_attention=False)
        self.enc3 = EncoderBlock(128,  256, use_attention=True)

        # fix: bottleneck should be ResBlock, not EncoderBlock (no downsampling here)
        self.bottleneck = ResBlock(256, 320)

        # fix: 2 args only (skip_ch removed — addition requires skip == out_ch anyway)
        self.dec0 = DecoderBlock(320, 256)
        self.dec1 = DecoderBlock(256, 128)
        self.dec2 = DecoderBlock(128,  64)
        self.dec3 = DecoderBlock(64,   32)

        # fix: added missing Linear(320→128); pool output is 320-dim from bottleneck
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(320, 128),   # fix: was missing
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 4),
        )

        # fix: nn.Conv3d (not bare Conv3d)
        self.seg_head = nn.Conv3d(32, 2, kernel_size=1)
        self.sdf_head = nn.Conv3d(32, 1, kernel_size=1)  # fix: 1 channel (scalar field)

        self.aux_head0 = nn.Conv3d(256, 2, kernel_size=1)
        self.aux_head1 = nn.Conv3d(128, 2, kernel_size=1)
        self.aux_head2 = nn.Conv3d(64,  2, kernel_size=1)

    def forward(self, x):
        target_size = x.shape[2:]

        x, skip0 = self.enc0(x)   # skip0: (B, 32,  64, 64, 64)  — no attention
        x, skip1 = self.enc1(x)   # skip1: (B, 64,  32, 32, 32)  — no attention
        x, skip2 = self.enc2(x)   # skip2: (B, 128, 16, 16, 16)  — MHSA applied
        x, skip3 = self.enc3(x)   # skip3: (B, 256,  8,  8,  8)  — MHSA applied

        x = self.bottleneck(x)    # (B, 320, 8, 8, 8)

        cls_out = self.cls_head(x)

        x = self.dec0(x, skip3)
        aux3 = F.interpolate(self.aux_head0(x), size=target_size,
                             mode='trilinear', align_corners=False)

        x = self.dec1(x, skip2)
        aux2 = F.interpolate(self.aux_head1(x), size=target_size,
                             mode='trilinear', align_corners=False)

        x = self.dec2(x, skip1)
        aux1 = F.interpolate(self.aux_head2(x), size=target_size,
                             mode='trilinear', align_corners=False)

        x = self.dec3(x, skip0)
        seg_out = self.seg_head(x)
        sdf_out = self.sdf_head(x)

        return seg_out, sdf_out, [aux1, aux2, aux3], cls_out

In [ ]:
model = BTUnetAttention().to(device)

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

NUM_EPOCHS = 150
SEG_LOSS_WEIGHT = 0.7
CLS_LOSS_WEIGHT = 0.3



from monai.losses import TverskyLoss, FocalLoss, HausdorffDTLoss

seg_loss_tv = TverskyLoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True,
    alpha=0.3,
    beta=0.7,
    batch=False
)

seg_loss_fc = FocalLoss(
    include_background=False,
    to_onehot_y=True,
    gamma=2.0,
    reduction='mean'
)

sdf_loss_fn = nn.HuberLoss(delta=0.1)

cls_loss_fn = nn.CrossEntropyLoss()

class EarlyStopping:
    def __init__(self, patience=7):
        self.patience = patience
        self.counter = 0
        self.best_dice = 0.0
        self.early_stop = False
    
    def __call__(self, val_dice):
        if val_dice > self.best_dice:
            self.best_dice = val_dice
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=10)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

val_dice_gli = DiceMetric(include_background=False, reduction="mean", num_classes=2, ignore_empty=True)
val_dice_men = DiceMetric(include_background=False, reduction="mean", num_classes=2, ignore_empty=True)
val_dice_meta = DiceMetric(include_background=False, reduction="mean", num_classes=2, ignore_empty=True)

train_dice_gli = DiceMetric(include_background=False, reduction="mean", num_classes=2, ignore_empty=False)
train_dice_men = DiceMetric(include_background=False, reduction="mean", num_classes=2, ignore_empty=False)
train_dice_meta = DiceMetric(include_background=False, reduction="mean", num_classes=2, ignore_empty=False)


# hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
post_pred = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)

best_dice = 0.0
best_cls_acc = 0.0
epoch_dice = 0.0
val_loss = 0

for epoch in range(NUM_EPOCHS):
    print(f"\n Epoch {epoch} / {NUM_EPOCHS}")

    model.train()
    train_dice_gli.reset()
    train_dice_men.reset()
    train_dice_meta.reset()
    epoch_seg_loss = 0.0
    epoch_cls_loss = 0.0
    cls_correct    = 0
    cls_total      = 0
    optimizer.zero_grad()
 
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False)

    for batch_idx, (images, masks, sdf_masks, tumor_types, images_128, masks_128) in enumerate(train_loop):
        images, masks, sdf_masks, tumor_types = images.to(device), masks.to(device), sdf_masks.to(device), tumor_types.to(device)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            seg_out, sdf_out, aux_outs, cls_out = model(images)
            seg_loss_t = seg_loss_tv(seg_out, masks)
            seg_loss_f = seg_loss_fc(seg_out, masks)
            seg_loss = ((0.6 * seg_loss_t) + (0.4 * seg_loss_f))
            cls_loss = cls_loss_fn(cls_out, tumor_types)

            aux_loss  = ((0.5  * ((0.6 * seg_loss_tv(aux_outs[0], masks)) + (0.4 * seg_loss_fc(aux_outs[0], masks)))) +
                         (0.25 * ((0.6 * seg_loss_tv(aux_outs[1], masks)) + (0.4 * seg_loss_fc(aux_outs[1], masks)))) +
                         (0.125* ((0.6 * seg_loss_tv(aux_outs[2], masks)) + (0.4 * seg_loss_fc(aux_outs[2], masks)))))

            sdf_loss  = sdf_loss_fn(sdf_out, sdf_masks)
            
            loss     = (SEG_LOSS_WEIGHT * (seg_loss + aux_loss + sdf_loss) +
                        CLS_LOSS_WEIGHT * cls_loss)  

            preds_list  = [post_pred(i) for i in decollate_batch(seg_out.detach())]
            labels_list = [post_label(i) for i in decollate_batch(masks)]
            
            for b, (pred_b, mask_b) in enumerate(zip(preds_list, labels_list)):
                t = tumor_types[b].item()
                if t == 0:
                    train_dice_gli(y_pred=[pred_b], y=[mask_b])
                elif t == 1:
                    train_dice_men(y_pred=[pred_b], y=[mask_b])
                elif t == 2:
                    train_dice_meta(y_pred=[pred_b], y=[mask_b]) 
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        with torch.no_grad():
            preds        = cls_out.argmax(dim=1)
            cls_correct += (preds == tumor_types).sum().item()
            cls_total   += tumor_types.size(0)
 
        epoch_seg_loss += seg_loss.item()
        epoch_cls_loss += cls_loss.item()
 
        train_loop.set_postfix({
            "seg": f"{seg_loss.item():.4f}",
            "cls": f"{cls_loss.item():.4f}",
        })

    train_dice_g, train_dice_m, train_dice_mt = train_dice_gli.aggregate().item(), train_dice_men.aggregate().item(), train_dice_meta.aggregate().item()
    mean_dice = 0.33 * (train_dice_g + train_dice_m + train_dice_mt)
    
    n = len(train_loader)
    seg_loss = epoch_seg_loss / n
    cls_loss = epoch_cls_loss / n
    train_cls_acc = cls_correct / cls_total if cls_total > 0 else 0.0
    train_mean_dice = mean_dice


    model.eval()
    # dice_metric.reset()
    val_dice_gli.reset()
    val_dice_men.reset()
    val_dice_meta.reset()
 
    val_cls_correct = 0
    val_cls_total   = 0
 
    val_loop = tqdm(val_loader, desc="Val", leave=False)
 
    with torch.no_grad():
        for images, masks, sdf_masks, tumor_types, images_128, masks_128 in val_loop:
            images = images.to(device)
            masks = masks.to(device)
            images_128 = images_128.to(device)
            masks_128 = masks_128.to(device)
            tumor_types = tumor_types.to(device)
 
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                # seg_out, cls_out = model(images)
                seg_out = sliding_window_inference(
                    inputs=images,
                    roi_size=(128, 128, 128),       # Must match your training crop size
                    sw_batch_size=4,             # Processes 4 sub-patches at a time on GPU
                    predictor=lambda x: model(x)[0],
                    overlap=0.25,                # 25% overlap for blending
                    mode="gaussian"              # Smooth blending weight map
                )

            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                seg, sdf, aux, cls_out = model(images_128)
 
            # segmentation metric
            outputs  = [post_pred(i)  for i in decollate_batch(seg_out)]
            masks_pp = [post_label(i) for i in decollate_batch(masks)]
            # dice_metric(y_pred=outputs, y=masks_pp)

            for b, (out_b, mask_b) in enumerate(zip(outputs, masks_pp)):
                t = tumor_types[b].item()
                if t == 0:   val_dice_gli(y_pred=[out_b], y=[mask_b])
                elif t == 1: val_dice_men(y_pred=[out_b], y=[mask_b])
                elif t == 2: val_dice_meta(y_pred=[out_b], y=[mask_b])        

            # classification accuracy
            preds = cls_out.argmax(dim=1)
            val_cls_correct += (preds == tumor_types).sum().item()
            val_cls_total += tumor_types.size(0)
    
            del images,images_128, masks, tumor_types, seg_out, cls_out, seg, aux, sdf
            del outputs, masks_pp
            torch.cuda.empty_cache()

    val_dice_g, val_dice_m, val_dice_mt = val_dice_gli.aggregate().item(), val_dice_men.aggregate().item(), val_dice_meta.aggregate().item()
    mean_dice = 0.33 * (val_dice_g + val_dice_m + val_dice_mt)
 
    # mean_dice    = dice_metric.aggregate().item()
    val_cls_acc  = val_cls_correct / val_cls_total \
                   if val_cls_total > 0 else 0.0
 
    scheduler.step()

    print(f"Seg Loss: {seg_loss:.4f} | Cls Loss: {cls_loss:.4f} | "
          f"Train Cls Acc: {train_cls_acc:.4f}")
    print(f"Training Dice - Glioma Dice: {train_dice_g} | Meningioma Dice: {train_dice_m} |  Metastatic Dice: {train_dice_mt} | Mean Dice: {train_mean_dice}")
    print(f"Val Dice: {mean_dice:.4f} | Glioma Dice: {val_dice_g} | Meningioma Dice: {val_dice_m} |  Metastatic Dice: {val_dice_mt} | Val Cls Acc: {val_cls_acc:.4f}")
    # save best segmentation model
    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save({
            'epoch'    : epoch,
            'model'    : model.module.state_dict()
                         if isinstance(model, torch.nn.DataParallel)
                         else model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_dice': best_dice,
        }, 'best_seg_checkpoint.pth')
        print(f"✅ Best seg model saved (Val Dice: {best_dice:.4f})")
 
    # save best classification model
    if val_cls_acc > best_cls_acc:
        best_cls_acc = val_cls_acc
        torch.save({
            'epoch'      : epoch,
            'model'      : model.module.state_dict()
                           if isinstance(model, torch.nn.DataParallel)
                           else model.state_dict(),
            'best_cls_acc': best_cls_acc,
        }, 'best_cls_checkpoint.pth')
        print(f"✅ Best cls model saved (Val Cls Acc: {best_cls_acc:.4f})")
 
    early_stopping(mean_dice)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break
 
    gc.collect()
 
# print(f"\nTraining complete. Best Val Dice: {best_dice:.4f}")
print(f"\nTraining complete. Best Val Dice: {best_dice:.4f} | "
      f"Best Val Cls Acc: {best_cls_acc:.4f}")

2

 Epoch 0 / 150


Epoch 0 Train:   0%|          | 0/300 [00:00<?, ?it/s]

Val:   0%|          | 0/150 [00:00<?, ?it/s]

Seg Loss: 0.5508 | Cls Loss: 0.6277 | Train Cls Acc: 0.7900
Training Dice - Glioma Dice: 0.20455709099769592 | Meningioma Dice: 0.11295927315950394 |  Metastatic Dice: 0.07341760396957397 | Mean Dice: 0.12900820948183536
Val Dice: 0.1356 | Glioma Dice: 0.2554415762424469 | Meningioma Dice: 0.07405690848827362 |  Metastatic Dice: 0.08128035813570023 | Val Cls Acc: 0.6867
✅ Best seg model saved (Val Dice: 0.1356)
✅ Best cls model saved (Val Cls Acc: 0.6867)

 Epoch 1 / 150


Epoch 1 Train:   0%|          | 0/300 [00:00<?, ?it/s]

...

Epoch 13 Train:   0%|          | 0/300 [00:00<?, ?it/s]

Val:   0%|          | 0/150 [00:00<?, ?it/s]

Seg Loss: 0.3121 | Cls Loss: 0.2419 | Train Cls Acc: 0.9283
Training Dice - Glioma Dice: 0.5271914005279541 | Meningioma Dice: 0.563471257686615 |  Metastatic Dice: 0.4421842694282532 | Mean Dice: 0.5058394861221314
Val Dice: 0.4297 | Glioma Dice: 0.5133663415908813 | Meningioma Dice: 0.37946465611457825 |  Metastatic Dice: 0.4093039631843567 | Val Cls Acc: 0.4600
✅ Best seg model saved (Val Dice: 0.4297)

 Epoch 14 / 150


Epoch 14 Train:   0%|          | 0/300 [00:00<?, ?it/s]

Val:   0%|          | 0/150 [00:00<?, ?it/s]